In [13]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Get the project root directory (where .env file should be)
# In Jupyter, getcwd() gives us the current working directory
project_root = Path(os.getcwd())
env_path = project_root / '.env'

# Debug: Check if .env file exists
if env_path.exists():
    print(f"Found .env file at: {env_path}")
else:
    print(f"WARNING: .env file not found at: {env_path}")
    print(f"Current working directory: {os.getcwd()}")

# Load the .env file with explicit path
load_dotenv(dotenv_path=env_path)

# Get the token
hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    print("ERROR: HF_TOKEN not found! Please check your .env file.")
    print("Make sure your .env file contains: HF_TOKEN=your_token_here")
else:
    print(f"Token loaded successfully (length: {len(hf_token)})")

Found .env file at: /Users/sunil/Developer/internship/valearnis_socrates/.env
Token loaded successfully (length: 37)


In [16]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
)

completion = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of France?"
        }
    ],
)

print(completion.choices[0].message)

ChatCompletionOutputMessage(role='assistant', content='The capital of France is Paris.', reasoning=None, tool_call_id=None, tool_calls=None)


In [ ]:
import os
from pathlib import Path
import file
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Load .env file before using environment variables (matching notebook Cell 0 pattern)
BASE_DIR = Path(__file__).resolve().parent.parent
env_path = BASE_DIR / ".env"
load_dotenv(dotenv_path=env_path)

def generate_response(user_input: str):
    """Generate a response using Hugging Face Inference API."""
    
    # Initialize client exactly like notebook Cell 1
    client = InferenceClient(
        api_key=os.environ["HF_TOKEN"],
    )
    
    system_prompt = """You are Socrates, the wise teacher for modern-day learners.
Do not provide direct answers. Instead, teach the user to think 
from first principles. End every response with a probing question."""

    # Primary: Try exact notebook pattern (user message only, no parameters)
    try:
        completion = client.chat.completions.create(
            model="meta-llama/Llama-3.1-8B-Instruct",
            messages=[
                {"role": "user", "content": f"{system_prompt}\n\nUser: {user_input}"},
            ],
        )
        return completion.choices[0].message.content
    except Exception as e:
        print(f"Chat completions (notebook pattern) failed: {str(e)[:200]}")
        
        # Fallback 1: Try text generation API (more reliable, doesn't use router)
        try:
            # Format prompt for Llama instruct models
            prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{user_input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
            response = client.text_generation(
                prompt=prompt,
                model="meta-llama/Llama-3.1-8B-Instruct",
                max_new_tokens=200,
                temperature=0.7,
                return_full_text=False,
                stop_sequences=["<|eot_id|>", "<|end_of_text|>"]
            )
            return response.strip()
        except Exception as e2:
            print(f"Text generation (Llama format) failed: {str(e2)[:200]}")
            
            # Fallback 2: Simple text generation
            try:
                simple_prompt = f"{system_prompt}\n\nUser: {user_input}\n\nAssistant:"
                response = client.text_generation(
                    prompt=simple_prompt,
                    model="meta-llama/Llama-3.1-8B-Instruct",
                    max_new_tokens=200,
                    temperature=0.7,
                    return_full_text=False
                )
                return response.strip()
            except Exception as e3:
                print(f"Simple text generation failed: {str(e3)[:200]}")
                
                # Fallback 3: Try alternative models with chat completions
                alternative_models = [
                    "mistralai/Mistral-7B-Instruct-v0.2",
                    "microsoft/Phi-3-mini-4k-instruct",
                ]
                
                for model in alternative_models:
                    try:
                        completion = client.chat.completions.create(
                            model=model,
                            messages=[
                                {"role": "user", "content": f"{system_prompt}\n\nUser: {user_input}"},
                            ],
                            max_tokens=200,
                            temperature=0.7
                        )
                        return completion.choices[0].message.content
                    except Exception as e4:
                        print(f"Alternative model {model} failed: {str(e4)[:200]}")
                        continue
                
                raise Exception(f"All methods failed. Chat: {str(e)[:100]}, TextGen1: {str(e2)[:100]}, TextGen2: {str(e3)[:100]}")

NameError: name '__file__' is not defined